# Modello 1: Deep Learning e Ottimizzazione al Primo Ordine

In questo modello esploriamo l'approccio "forza bruta" e ad alta dimensionalità. Il modello riceve in input l'intera scacchiera codificata in un vettore sparso $\mathbf{x} \in \mathbb{R}^{768}$.

## L'Architettura (Algebra Lineare)
Il nostro modello è una Rete Neurale *Feedforward*. Ogni passaggio da uno strato all'altro è una trasformazione affine seguita da una funzione di attivazione non lineare (ReLU). Matematicamente, il passaggio dal livello $l$ al livello $l+1$ è definito come:

$$\mathbf{z}^{(l+1)} = W^{(l)} \mathbf{a}^{(l)} + \mathbf{b}^{(l)}$$
$$\mathbf{a}^{(l+1)} = \text{ReLU}(\mathbf{z}^{(l+1)})$$

Dove $W$ è la matrice dei pesi di quello strato e $\mathbf{b}$ è il vettore dei bias.

## Ottimizzazione (Steepest Descent)
Poiché il numero totale di parametri (pesi) in questa architettura ammonta a centinaia di migliaia, il calcolo della Matrice Hessiana $H$ è computazionalmente proibitivo. Pertanto, ci affidiamo a un'ottimizzazione al primo ordine (Discesa del Gradiente).
L'aggiornamento dei pesi avviene calcolando la derivata della Funzione di Costo $L$ (Errore Quadratico Medio) rispetto al vettore dei pesi $\mathbf{w}$:

$$\mathbf{w}_{new} = \mathbf{w}_{old} - \alpha \nabla L(\mathbf{w}_{old})$$

Dove $\alpha$ è il nostro *Learning Rate* (passo) e $\nabla L$ è il vettore Gradiente calcolato tramite l'algoritmo di *Backpropagation*.

In [1]:
# ==========================================
# CELLA 2: CARICAMENTO DATI E TRASFORMAZIONE NON-LINEARE
# ==========================================
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

print("Caricamento dati in corso...")
dati = np.load('dataset_elaborato.npz')

X_numpy = dati['X_768']
Y_numpy = dati['Y'].copy() # Usiamo copy() per lavorare in sicurezza sui dati

# ---------------------------------------------------------
# IL TOCCO MAGICO DELLA DATA SCIENCE
# ---------------------------------------------------------
# 1. Convertiamo i centipedoni in pedoni (es. -33 diventa -0.33)
Y_numpy = Y_numpy / 100.0

# 2. Applichiamo la Tangente Iperbolica: Y = tanh(Y / 4.0)
# Questo trasforma le valutazioni in "Probabilità di Vittoria" tra -1 e +1.
# I matti (+85.0 pedoni) diventeranno +1.0 (100% vittoria Bianco).
Y_numpy = np.tanh(Y_numpy / 4.0)

# ---------------------------------------------------------

# 3. Creiamo i Tensori per PyTorch
X_tensor = torch.tensor(X_numpy, dtype=torch.float32)
Y_tensor = torch.tensor(Y_numpy, dtype=torch.float32).view(-1, 1) 

# 4. Impacchettiamo in lotti (Batch)
dimensione_batch = 512
dataset = TensorDataset(X_tensor, Y_tensor)
dataloader = DataLoader(dataset, batch_size=dimensione_batch, shuffle=True)

print(f"✅ Dati pronti! Target trasformato con successo in probabilità [-1, 1].")
print(f"   Valore Minimo: {Y_tensor.min().item():.3f} (Vantaggio decisivo Nero)")
print(f"   Valore Massimo: {Y_tensor.max().item():.3f} (Vantaggio decisivo Bianco)")
print(f"   Valore Medio: {Y_tensor.mean().item():.3f}")

Caricamento dati in corso...
✅ Dati pronti! Target trasformato con successo in probabilità [-1, 1].
   Valore Minimo: -1.000 (Vantaggio decisivo Nero)
   Valore Massimo: 1.000 (Vantaggio decisivo Bianco)
   Valore Medio: 0.045


In [2]:
# Definiamo l'architettura della nostra Rete Neurale
class MotoreScacchiNN(nn.Module):
    def __init__(self):
        super(MotoreScacchiNN, self).__init__()
        
        # Livello 1: Prende i 768 input e li collega a 256 neuroni nascosti
        self.strato_1 = nn.Linear(768, 256)
        self.attivazione_1 = nn.ReLU() # La non-linearità che permette di imparare concetti complessi
        
        # Livello 2: Da 256 a 32 neuroni (restringiamo l'imbuto)
        self.strato_2 = nn.Linear(256, 32)
        self.attivazione_2 = nn.ReLU()
        
        # Livello 3 (Output): Da 32 neuroni a 1 singolo numero (la valutazione!)
        self.strato_output = nn.Linear(32, 1)

    def forward(self, x):
        # Definiamo come i dati viaggiano in avanti (Forward Pass)
        x = self.strato_1(x)
        x = self.attivazione_1(x)
        
        x = self.strato_2(x)
        x = self.attivazione_2(x)
        
        x = self.strato_output(x)
        return x

# Creiamo "fisicamente" il modello nella memoria del PC
modello_1 = MotoreScacchiNN()

# Definiamo l'obiettivo matematico (Funzione di Costo): Errore Quadratico Medio (MSE)
funzione_costo = nn.MSELoss()

# Definiamo l'algoritmo di Ottimizzazione al primo ordine.
# Usiamo Adam (una versione più intelligente e stabile della pura Discesa del Gradiente)
learning_rate = 0.001
ottimizzatore = optim.Adam(modello_1.parameters(), lr=learning_rate)

print("Modello 1 istanziato! Architettura:")
print(modello_1)

Modello 1 istanziato! Architettura:
MotoreScacchiNN(
  (strato_1): Linear(in_features=768, out_features=256, bias=True)
  (attivazione_1): ReLU()
  (strato_2): Linear(in_features=256, out_features=32, bias=True)
  (attivazione_2): ReLU()
  (strato_output): Linear(in_features=32, out_features=1, bias=True)
)


## Il Ciclo di Addestramento (Training Loop) e Backpropagation

L'addestramento procede per iterazioni chiamate **Epoche** (una scansione completa del dataset). All'interno di ogni epoca, i dati vengono processati in **Lotti (Batches)** per ottimizzare l'uso della memoria e stabilizzare la discesa del gradiente.

Per ogni lotto, l'algoritmo esegue le seguenti operazioni matematiche:
1. **Forward Pass:** Calcola $\mathbf{\hat{y}} = f(\mathbf{x}; \mathbf{w})$.
2. **Loss Computation:** Misura l'errore tramite la funzione $L(\mathbf{\hat{y}}, \mathbf{y}) = \frac{1}{n}\sum(\hat{y}_i - y_i)^2$.
3. **Backward Pass:** Applica la *Regola della Catena* (Chain Rule) per calcolare il gradiente della Loss rispetto a ogni singolo peso della rete: $\frac{\partial L}{\partial w_{ij}}$.
4. **Optimization Step:** Aggiorna il vettore dei pesi $\mathbf{w}$ muovendosi nella direzione opposta al gradiente.

In PyTorch, questo processo elegante si traduce in tre semplici comandi: `zero_grad()`, `backward()`, e `step()`.

In [3]:
# ==========================================
# CELLA 5: IL CICLO DI ADDESTRAMENTO E BACKPROPAGATION
# ==========================================
from tqdm import tqdm

# Definiamo quante volte il modello vedrà l'intero dataset (Epoche)
# Per iniziare, 10 epoche sono un buon compromesso tra tempo e apprendimento
epoche = 10 
storico_loss = [] # Teniamo traccia di come scende l'errore

print("🚀 Inizio dell'addestramento (Ottimizzazione al 1° Ordine in azione!)...")

for epoca in range(epoche):
    loss_totale_epoca = 0.0
    
    # Iteriamo sui lotti (batches) creati in precedenza
    # Usiamo tqdm per avere la barra di caricamento per ogni epoca
    loop_lotti = tqdm(dataloader, desc=f"Epoca {epoca+1}/{epoche}")
    
    for batch_X, batch_Y in loop_lotti:
        
        # 1. AZZERAMENTO GRADIENTI
        # PyTorch accumula le derivate di default. Dobbiamo azzerarle a ogni step.
        ottimizzatore.zero_grad()
        
        # 2. FORWARD PASS
        # Il modello guarda le 512 scacchiere e spara le sue valutazioni
        previsioni = modello_1(batch_X)
        
        # 3. CALCOLO DELL'ERRORE (Loss)
        # Distanza quadratica media (MSE) tra previsioni e valutazioni Stockfish
        errore = funzione_costo(previsioni, batch_Y)
        
        # 4. BACKWARD PASS (Il miracolo del calcolo differenziale)
        # Calcola le derivate parziali per tutti i 200.000+ pesi contemporaneamente
        errore.backward()
        
        # 5. AGGIORNAMENTO PESI (Steepest Descent / Adam)
        # Modifica la matrice W sottraendo una frazione del gradiente
        ottimizzatore.step()
        
        # Sommiamo l'errore per calcolare la media a fine epoca
        loss_totale_epoca += errore.item()
        
        # Aggiorniamo il numerino sulla barra di caricamento in tempo reale
        loop_lotti.set_postfix(Loss=errore.item())
        
    # Calcoliamo la loss media per tutta l'epoca
    loss_media = loss_totale_epoca / len(dataloader)
    storico_loss.append(loss_media)
    print(f"✅ Epoca {epoca+1} completata! Errore Medio (MSE): {loss_media:.4f}\n")

# ==========================================
# SALVATAGGIO DEL MODELLO (IL CHECKPOINT)
# ==========================================
percorso_salvataggio = 'pesi_modello_1_deep_learning.pth'
torch.save(modello_1.state_dict(), percorso_salvataggio)

print(f"🎉 Addestramento Modello 1 terminato! I pesi ottimizzati sono stati salvati in '{percorso_salvataggio}'.")

🚀 Inizio dell'addestramento (Ottimizzazione al 1° Ordine in azione!)...


Epoca 1/10: 100%|██████████| 659/659 [00:09<00:00, 70.13it/s, Loss=0.393]


✅ Epoca 1 completata! Errore Medio (MSE): 0.3668



Epoca 2/10: 100%|██████████| 659/659 [00:09<00:00, 71.30it/s, Loss=0.45] 


✅ Epoca 2 completata! Errore Medio (MSE): 0.3660



...

✅ Epoca 8 completata! Errore Medio (MSE): 0.3568



Epoca 9/10: 100%|██████████| 659/659 [00:10<00:00, 64.02it/s, Loss=0.261]


✅ Epoca 9 completata! Errore Medio (MSE): 0.3540



Epoca 10/10: 100%|██████████| 659/659 [00:10<00:00, 64.97it/s, Loss=0.532]


✅ Epoca 10 completata! Errore Medio (MSE): 0.3515

🎉 Addestramento Modello 1 terminato! I pesi ottimizzati sono stati salvati in 'pesi_modello_1_deep_learning.pth'.
